# IMQCAM Export — Select Form, Then IGSN(s)

Researcher workflow:

1. Select the IMQCAM form
2. Enter a parent IGSN or one/more child IGSNs
3. Run the notebook
4. Export a clean Excel workbook

### Matching rules

- Parent IGSN such as `CMXMAG00002`
  - returns the parent and all matching child IGSNs from the selected form

- Child IGSN such as `CMXMAG00002-044`
  - returns only that exact child

- Multiple child IGSNs
  - returns only those selected children

### Form-aware export

The notebook automatically normalizes important repeated structures:

- **AM Build Parameters** → one row per IGSN × build region
- **Sample Heat Treatment Record** → one row per IGSN × thermal stage
- **Four-point flexural test** → one row per IGSN × test
- **Fractography Record** → one row per IGSN × defect/test record
- **Raw Powder Details** → one row per IGSN × composition record when composition is repeated
- **Printer Build** → clean build metadata plus a separate Build Geometries sheet
- **NASA Ti64 30um Layers Data** → flattened columns
- **Archival ULI Build (simplified)** → flattened columns
- **Micromechanical Simulation Data** → flattened columns, with nested values preserved readably

This notebook is read-only.

## Connect to IMQCAM

The notebook connects directly to the IMQCAM REST API.

In [ ]:
import json
import os
import re
from getpass import getpass
from pathlib import Path

import pandas as pd

try:
    from girder_client import GirderClient
except ImportError:
    raise ImportError(
        "girder-client is not installed. Run: "
        "%pip install girder-client pandas xlsxwriter openpyxl"
    )

API_URL = "https://data.imqcam.org/api/v1"

api_key = os.environ.get("GIRDER_API_KEY")

if not api_key:
    api_key = getpass("Enter IMQCAM GIRDER_API_KEY: ")

client = GirderClient(apiUrl=API_URL)
client.authenticate(apiKey=api_key)

user = client.get("user/me")

print("Connected to:", client.urlBase)
print("Authenticated as:", user.get("login") or user.get("email"))

## List available forms

Run this cell if you want to see the exact form names available in IMQCAM.

In [ ]:
forms = client.get("form", parameters={"limit": 1000})

forms_df = pd.DataFrame([
    {
        "Form Name": f.get("name"),
        "Form ID": f.get("_id"),
    }
    for f in forms
]).sort_values("Form Name").reset_index(drop=True)

display(forms_df)

In [ ]:
FORM_NAME = "AM Build Parameters"

IGSNS = [
    "CMXMAG00002",
]

WORKSPACE_ROOT = next(
    (directory for directory in (Path.cwd(), *Path.cwd().parents) if (directory / ".git").exists()),
    Path.cwd(),
)
OUTPUT_DIRECTORY = WORKSPACE_ROOT / "downloads"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIRECTORY / "IMQCAM_Selected_Form_IGSN_Export.xlsx"

print("Form:", FORM_NAME)
print("IGSN query:", IGSNS)

## Find the selected form

In [ ]:
form_lookup = {
    f.get("name"): f.get("_id")
    for f in forms
}

if FORM_NAME not in form_lookup:
    raise ValueError(
        f"Form '{FORM_NAME}' was not found. "
        "Choose an exact form name from the table above."
    )

FORM_ID = form_lookup[FORM_NAME]

print("Selected form:", FORM_NAME)
print("Form ID:", FORM_ID)

## Retrieve entries from only the selected form

The notebook does not search every form.  
It first restricts the API query to the selected `FORM_NAME`, then applies the IGSN filter.

In [ ]:
IGSN_PATTERN = re.compile(r"\b[A-Z]{6}\d{5}(?:-\d+)?\b")


def normalize_igsns(values):
    result = []

    for value in values:
        if not isinstance(value, str):
            continue

        for match in IGSN_PATTERN.findall(value.upper()):
            if match not in result:
                result.append(match)

    return result


QUERY_IGSNS = normalize_igsns(IGSNS)

if not QUERY_IGSNS:
    raise ValueError("No valid IGSNs were provided.")


def extract_igsns(value):
    found = []

    def walk(obj):
        if isinstance(obj, dict):
            for v in obj.values():
                walk(v)

        elif isinstance(obj, list):
            for v in obj:
                walk(v)

        elif isinstance(obj, str):
            for match in IGSN_PATTERN.findall(obj.upper()):
                if match not in found:
                    found.append(match)

    walk(value)
    return found


def is_parent(igsn):
    return "-" not in igsn


def get_matches(entry_igsns):
    matches = []

    for query in QUERY_IGSNS:

        if is_parent(query):
            current = [
                igsn
                for igsn in entry_igsns
                if igsn == query or igsn.startswith(query + "-")
            ]

        else:
            current = [
                igsn
                for igsn in entry_igsns
                if igsn == query
            ]

        for igsn in current:
            if igsn not in matches:
                matches.append(igsn)

    return matches


def fetch_form_entries(form_id, page_size=100):
    entries = []
    offset = 0

    while True:
        page = client.get(
            "entry",
            parameters={
                "formId": form_id,
                "limit": page_size,
                "offset": offset,
                "sort": "created",
                "sortdir": 1,
            },
        )

        entries.extend(page)

        if len(page) < page_size:
            break

        offset += len(page)

    return entries


form_entries = fetch_form_entries(FORM_ID)

matched_entries = []

for entry in form_entries:
    data = entry.get("data") or {}
    entry_igsns = extract_igsns(data)
    matches = get_matches(entry_igsns)

    if matches:
        matched_entries.append({
            "entry": entry,
            "matched_igsns": matches,
            "all_entry_igsns": entry_igsns,
        })

print("Selected form:", FORM_NAME)
print("Query IGSNs:", QUERY_IGSNS)
print("Entries in selected form:", len(form_entries))
print("Matching entries:", len(matched_entries))

## Create clean researcher-friendly results

The selected form determines how the export is structured.

The notebook keeps the main result table easy to analyze and expands important repeated arrays into rows instead of placing large Python/JSON objects into a single cell.

In [ ]:
def normalize_for_excel(value):
    """Convert any remaining nested value into readable JSON."""
    if isinstance(value, (dict, list, tuple, set)):
        return json.dumps(value, ensure_ascii=False, default=str)
    return value


def matched_lookup_igsns(data, fallback_igsns):
    """
    Prefer a form's lookup array when available.
    Return only IGSNs relevant to the current query.
    """
    lookup = data.get("lookup")
    if lookup is None:
        return fallback_igsns

    lookup_igsns = extract_igsns(lookup)
    matches = get_matches(lookup_igsns)
    return matches or fallback_igsns


def base_entry_fields(entry, igsn):
    return {
        "IGSN": igsn,
        "Entry ID": entry.get("_id"),
        "Unique ID": entry.get("uniqueId"),
        "Created": entry.get("created"),
        "Updated": entry.get("updated"),
    }


def flatten_dict(prefix, obj):
    """Flatten one dictionary into dotted keys."""
    if not isinstance(obj, dict):
        return {}
    rows = pd.json_normalize(obj, sep=".").to_dict("records")
    return rows[0] if rows else {}


def make_generic_rows(item, skip_fields=None):
    """
    Generic one-row-per-matched-IGSN representation.
    Dicts flatten into dotted columns; remaining arrays are readable JSON.
    """
    skip_fields = set(skip_fields or [])
    entry = item["entry"]
    data = entry.get("data") or {}
    flat = flatten_dict("", data)

    rows = []

    for igsn in item["matched_igsns"]:
        row = base_entry_fields(entry, igsn)

        for key, value in flat.items():
            root_key = key.split(".")[0]

            if root_key in skip_fields:
                continue

            row[key] = normalize_for_excel(value)

        rows.append(row)

    return rows


extra_tables = {}


# ============================================================
# AM BUILD PARAMETERS
# ============================================================
if FORM_NAME == "AM Build Parameters":

    rows = []

    for item in matched_entries:
        entry = item["entry"]
        data = entry.get("data") or {}

        relevant_igsns = matched_lookup_igsns(
            data,
            item["matched_igsns"],
        )

        params = data.get("buildParameters") or []
        if not isinstance(params, list):
            params = []

        for igsn in relevant_igsns:
            for param in params:
                if not isinstance(param, dict):
                    continue

                is_pulsed = (
                    param.get("pulseDistance") is not None
                    or param.get("exposureTime") is not None
                )

                rows.append({
                    "IGSN": igsn,
                    "Entry ID": entry.get("_id"),
                    "Unique ID": data.get("uniqueId") or entry.get("uniqueId"),
                    "Region Type": param.get("type"),
                    "Mode": "Pulsed" if is_pulsed else "Continuous",
                    "Laser Speed": param.get("laserSpeed"),
                    "Pulse Distance": param.get("pulseDistance"),
                    "Exposure Time": param.get("exposureTime"),
                    "Laser Power": param.get("laserPower"),
                    "Layer Thickness": param.get("layerThickness"),
                    "Hatch Spacing": param.get("hatchSpacing"),
                    "Created": entry.get("created"),
                    "Updated": entry.get("updated"),
                })

    results_df = pd.DataFrame(rows)


# ============================================================
# SAMPLE HEAT TREATMENT RECORD
# ============================================================
elif FORM_NAME == "Sample Heat Treatment Record":

    rows = []

    for item in matched_entries:
        entry = item["entry"]
        data = entry.get("data") or {}

        relevant_igsns = matched_lookup_igsns(
            data,
            item["matched_igsns"],
        )

        stress = data.get("stress_relief") or {}
        cycles = data.get("heat_treatment") or []

        if not isinstance(cycles, list):
            cycles = []

        for igsn in relevant_igsns:
            rows.append({
                "IGSN": igsn,
                "Entry ID": entry.get("_id"),
                "Heat Treatment ID": data.get("heat_treatment_id"),
                "Stage": "Stress Relief",
                "Cycle #": 0,
                "Time (hr)": stress.get("time_hr"),
                "Temperature (C)": stress.get("temperature_C"),
                "Ramp Up (C/min)": stress.get("ramp_up_C_per_min"),
                "Cooling Method": stress.get("cooling_method"),
                "Environment": stress.get("environment"),
                "HIP": "No",
                "HIP Pressure (MPa)": None,
                "Related Files": normalize_for_excel(stress.get("files")),
                "Created": entry.get("created"),
                "Updated": entry.get("updated"),
            })

            for cycle_no, cycle in enumerate(cycles, start=1):
                if not isinstance(cycle, dict):
                    continue

                hip = cycle.get("HIP")
                is_hip = isinstance(hip, dict)

                rows.append({
                    "IGSN": igsn,
                    "Entry ID": entry.get("_id"),
                    "Heat Treatment ID": data.get("heat_treatment_id"),
                    "Stage": "Heat Treatment",
                    "Cycle #": cycle_no,
                    "Time (hr)": cycle.get("time_hr"),
                    "Temperature (C)": cycle.get("temperature_C"),
                    "Ramp Up (C/min)": cycle.get("ramp_up_C_per_min"),
                    "Cooling Method": cycle.get("cooling_method"),
                    "Environment": cycle.get("environment"),
                    "HIP": "Yes" if is_hip else "No",
                    "HIP Pressure (MPa)": hip.get("pressure_MPa") if is_hip else None,
                    "Related Files": normalize_for_excel(cycle.get("files")),
                    "Created": entry.get("created"),
                    "Updated": entry.get("updated"),
                })

    results_df = pd.DataFrame(rows)


# ============================================================
# FOUR-POINT FLEXURAL TEST
# ============================================================
elif FORM_NAME == "Four-point flexural test":

    rows = []

    for item in matched_entries:
        entry = item["entry"]
        data = entry.get("data") or {}
        tests = data.get("tests") or []

        if not isinstance(tests, list):
            tests = []

        parent_flat = flatten_dict("", {
            k: v for k, v in data.items()
            if k not in {"tests", "lookup"}
        })

        for igsn in item["matched_igsns"]:
            if not tests:
                row = base_entry_fields(entry, igsn)
                row.update({k: normalize_for_excel(v) for k, v in parent_flat.items()})
                rows.append(row)
                continue

            for test_no, test in enumerate(tests, start=1):
                if not isinstance(test, dict):
                    continue

                row = base_entry_fields(entry, igsn)
                row.update({k: normalize_for_excel(v) for k, v in parent_flat.items()})
                row["Test Row #"] = test_no

                for key, value in flatten_dict("", test).items():
                    row[key] = normalize_for_excel(value)

                rows.append(row)

    results_df = pd.DataFrame(rows)


# ============================================================
# FRACTOGRAPHY RECORD
# ============================================================
elif FORM_NAME == "Fractography Record":

    rows = []

    for item in matched_entries:
        entry = item["entry"]
        data = entry.get("data") or {}
        tests = data.get("tests") or []

        if not isinstance(tests, list):
            tests = []

        parent_flat = flatten_dict("", {
            k: v for k, v in data.items()
            if k not in {"tests", "lookup"}
        })

        for igsn in item["matched_igsns"]:
            if not tests:
                row = base_entry_fields(entry, igsn)
                row.update({k: normalize_for_excel(v) for k, v in parent_flat.items()})
                rows.append(row)
                continue

            for record_no, test in enumerate(tests, start=1):
                if not isinstance(test, dict):
                    continue

                row = base_entry_fields(entry, igsn)
                row.update({k: normalize_for_excel(v) for k, v in parent_flat.items()})
                row["Record #"] = record_no

                for key, value in flatten_dict("", test).items():
                    row[key] = normalize_for_excel(value)

                rows.append(row)

    results_df = pd.DataFrame(rows)


# ============================================================
# RAW POWDER DETAILS
# ============================================================
elif FORM_NAME == "Raw Powder Details":

    rows = []
    composition_rows = []

    for item in matched_entries:
        entry = item["entry"]
        data = entry.get("data") or {}

        # Main powder metadata without the potentially repeated composition list.
        rows.extend(
            make_generic_rows(
                item,
                skip_fields={"composition"},
            )
        )

        composition = data.get("composition") or []
        if isinstance(composition, list):
            for igsn in item["matched_igsns"]:
                for component_no, component in enumerate(composition, start=1):
                    if not isinstance(component, dict):
                        continue

                    row = base_entry_fields(entry, igsn)
                    row["Component #"] = component_no

                    for key, value in flatten_dict("", component).items():
                        row[key] = normalize_for_excel(value)

                    composition_rows.append(row)

    results_df = pd.DataFrame(rows)

    if composition_rows:
        extra_tables["Composition"] = pd.DataFrame(composition_rows)


# ============================================================
# PRINTER BUILD
# ============================================================
elif FORM_NAME == "Printer Build":

    rows = []
    geometry_rows = []
    misc_rows = []

    for item in matched_entries:
        entry = item["entry"]
        data = entry.get("data") or {}

        # Do not export legacy Printer Build buildParameters.
        # AM Build Parameters is the dedicated process-parameter form.
        rows.extend(
            make_generic_rows(
                item,
                skip_fields={
                    "buildParameters",
                    "buildGeometries",
                    "misc",
                    "igsn",
                    "igsnMeta",
                },
            )
        )

        geometries = data.get("buildGeometries") or []
        if isinstance(geometries, list):
            for igsn in item["matched_igsns"]:
                for n, geometry in enumerate(geometries, start=1):
                    if not isinstance(geometry, dict):
                        continue

                    row = base_entry_fields(entry, igsn)
                    row["Geometry #"] = n
                    row.update({
                        k: normalize_for_excel(v)
                        for k, v in flatten_dict("", geometry).items()
                    })
                    geometry_rows.append(row)

        misc = data.get("misc") or []
        if isinstance(misc, list):
            for igsn in item["matched_igsns"]:
                for n, value in enumerate(misc, start=1):
                    if not isinstance(value, dict):
                        continue

                    row = base_entry_fields(entry, igsn)
                    row["Metadata #"] = n
                    row.update({
                        k: normalize_for_excel(v)
                        for k, v in flatten_dict("", value).items()
                    })
                    misc_rows.append(row)

    results_df = pd.DataFrame(rows)

    if geometry_rows:
        extra_tables["Build Geometries"] = pd.DataFrame(geometry_rows)

    if misc_rows:
        extra_tables["Misc Metadata"] = pd.DataFrame(misc_rows)


# ============================================================
# NASA Ti64 / ARCHIVAL ULI / MICROMECHANICAL / OTHER FORMS
# ============================================================
else:

    rows = []

    for item in matched_entries:
        rows.extend(
            make_generic_rows(
                item,
                skip_fields={"lookup"},
            )
        )

    results_df = pd.DataFrame(rows)


# Tidy ordering where possible.
if not results_df.empty:
    preferred = [
        "IGSN",
        "Entry ID",
        "Unique ID",
        "Created",
        "Updated",
    ]

    ordered = [
        col for col in preferred
        if col in results_df.columns
    ] + [
        col for col in results_df.columns
        if col not in preferred
    ]

    results_df = results_df[ordered].reset_index(drop=True)


print("Selected form:", FORM_NAME)
print("Main export rows:", len(results_df))

if extra_tables:
    print("Additional sheets:", ", ".join(extra_tables.keys()))

display(results_df)

## Export clean results to Excel

The workbook always contains a `Results` sheet.

Forms with meaningful repeated structures can also produce additional sheets such as:

- `Build Geometries`
- `Composition`
- `Misc Metadata`

This keeps the main researcher table clean while preserving useful repeated metadata.

In [ ]:
try:
    import xlsxwriter
    engine = "xlsxwriter"
except ImportError:
    engine = "openpyxl"


def safe_sheet_name(name, used):
    clean = re.sub(r"[\[\]:*?/\\]", "_", str(name))[:31] or "Sheet"
    candidate = clean
    n = 2

    while candidate in used:
        suffix = f"_{n}"
        candidate = clean[:31-len(suffix)] + suffix
        n += 1

    used.add(candidate)
    return candidate


tables = {"Results": results_df}
tables.update(extra_tables)

used = set()

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine=engine,
) as writer:

    written = {}

    for requested_sheet, df in tables.items():
        sheet = safe_sheet_name(requested_sheet, used)
        df.to_excel(
            writer,
            sheet_name=sheet,
            index=False,
        )
        written[sheet] = df

    if engine == "xlsxwriter":
        workbook = writer.book

        header_fmt = workbook.add_format({
            "bold": True,
            "font_color": "white",
            "bg_color": "#1F4E78",
            "border": 1,
            "align": "center",
            "valign": "vcenter",
        })

        wrap_fmt = workbook.add_format({
            "text_wrap": True,
            "valign": "top",
        })

        for sheet, df in written.items():
            worksheet = writer.sheets[sheet]
            worksheet.freeze_panes(1, 0)

            for col_num, col in enumerate(df.columns):
                worksheet.write(0, col_num, col, header_fmt)

            if len(df) > 0 and len(df.columns) > 0:
                worksheet.autofilter(
                    0,
                    0,
                    len(df),
                    len(df.columns) - 1,
                )

            for col_num, col in enumerate(df.columns):
                values = [str(col)]

                if len(df):
                    values.extend(
                        str(v)
                        for v in df[col].head(200).fillna("")
                    )

                width = min(
                    max(len(v) for v in values) + 2,
                    42,
                )
                width = max(width, 12)

                long_text = any(
                    token in str(col).lower()
                    for token in [
                        "file",
                        "comment",
                        "description",
                        "notes",
                        "metadata",
                    ]
                )

                worksheet.set_column(
                    col_num,
                    col_num,
                    42 if long_text else width,
                    wrap_fmt if long_text else None,
                )


print("Excel created:", OUTPUT_FILE)
print("Sheets:", ", ".join(tables.keys()))

## Examples

### AM Build Parameters

```python
FORM_NAME = "AM Build Parameters"
IGSNS = ["CMXMAG00002"]
```

Produces one row per matched specimen × scan region.

### Four-point flexural test

```python
FORM_NAME = "Four-point flexural test"
IGSNS = ["CMXMAL00011"]
```

Produces one row per matched specimen × fatigue test.

### Fractography Record

```python
FORM_NAME = "Fractography Record"
IGSNS = ["CMXMAL00011-001"]
```

Produces one row per repeated fractography/defect record.

### Printer Build

```python
FORM_NAME = "Printer Build"
IGSNS = ["CMXMAG00002"]
```

Produces:
- `Results` — build metadata
- `Build Geometries` — one row per geometry
- `Misc Metadata` when present

Legacy `buildParameters` inside Printer Build are intentionally omitted because AM process parameters belong in the dedicated **AM Build Parameters** form.

### Sample Heat Treatment Record

```python
FORM_NAME = "Sample Heat Treatment Record"
IGSNS = ["CMXMAL00025"]
```

Produces one row per IGSN × thermal stage.

### Other forms

NASA Ti64, Archival ULI, and Micromechanical Simulation Data are flattened into ordinary Excel columns while preserving any remaining nested values readably.